In [ ]:
# CELL 1: Setup PATH and Imports
import os
import subprocess
import numpy as np
import nibabel as nib
from pathlib import Path
import psutil
import scipy.io as sio
from scipy.stats import spearmanr, pearsonr,kendalltau
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics.pairwise import cosine_similarity 
import pandas as pd
# Add Connectome Workbench to PATH
os.environ['PATH'] = '/opt/workbench/bin_linux64:' + os.environ['PATH']

# Verify 
try:
    subprocess.run(['wb_command', '-version'], capture_output=True, check=True)
    print("✓ wb_command found and working")
except FileNotFoundError:
    print("✗ wb_command not found - check installation")

In [ ]:
# ============================================================================
# TIMING AND SEGMENTATION FUNCTIONS
# ============================================================================

def average_2s_bins_delay(X, TR, onset_sec, duration_sec, bin_sec=2.0, delay_sec=4.0):
    """
    Extract and bin fMRI data with BOLD delay compensation.
    
    Parameters:
    -----------
    X : np.ndarray
        fMRI data (n_vertices, n_timepoints)
    TR : float
        Repetition time in seconds
    onset_sec : float
        Stimulus onset in seconds
    duration_sec : float
        Stimulus duration in seconds
    bin_sec : float
        Bin size in seconds (default 2.0)
    delay_sec : float
        BOLD delay in seconds (default 4.0)

    Returns:
    --------
    Y : np.ndarray
        Binned data (n_vertices, n_bins)
    """
    n_vertices, n_timepoints = X.shape

    # Apply delay to the stimulus window
    onset_d = onset_sec + delay_sec
    end_d = onset_sec + duration_sec + delay_sec

    # Convert to timepoint indices
    start_tp = int(np.floor(onset_d / TR))
    end_tp = int(np.ceil(end_d / TR))

    # Clip to valid range
    start_tp = max(start_tp, 0)
    end_tp = min(end_tp, n_timepoints)

    if end_tp <= start_tp:
        return np.zeros((n_vertices, 0), dtype=X.dtype)

    segment = X[:, start_tp:end_tp]

    # Timepoints per bin
    tp_per_bin = int(np.round(bin_sec / TR))
    if tp_per_bin < 1:
        raise ValueError(f"bin_sec/TR is < 1 (bin_sec={bin_sec}, TR={TR})")

    # Drop leftover timepoints that don't fill a full bin
    n_bins = segment.shape[1] // tp_per_bin
    segment = segment[:, :n_bins * tp_per_bin]

    if n_bins == 0:
        return np.zeros((n_vertices, 0), dtype=X.dtype)

    Y = segment.reshape(n_vertices, n_bins, tp_per_bin).mean(axis=2)
    return Y


def process_all_video_segments(fmri_data, timing_df, TR=1.0, bin_sec=2.0, delay_sec=4.0):
    """
    Process all video segments according to timing file.
    
    Parameters:
    -----------
    fmri_data : np.ndarray
        Full fMRI data (n_vertices, n_timepoints)
    timing_df : pd.DataFrame
        DataFrame with columns: video_id, onset_sec, duration_sec
    TR : float
        Repetition time in seconds
    bin_sec : float
        Bin size in seconds
    delay_sec : float
        BOLD delay in seconds
    
    Returns:
    --------
    processed_fmri : np.ndarray
        Concatenated binned data (n_vertices, total_bins)
    """
    import pandas as pd
    
    segments = []
    
    for _, row in timing_df.iterrows():
        binned = average_2s_bins_delay(
            fmri_data, 
            TR=TR,
            onset_sec=row["onset_sec"],
            duration_sec=row["duration_sec"],
            bin_sec=bin_sec,
            delay_sec=delay_sec
        )
        segments.append(binned)
        print(f"  Video {row['video_id']}: {binned.shape[1]} bins")
    
    processed_fmri = np.concatenate(segments, axis=1)
    return processed_fmri

In [ ]:
def convert_mat_to_gifti(mat_file, data_key, output_gifti, vertices_per_hemisphere=32492):
    """
    Convert .mat file with movie-watching data to GIFTI format.
    
    Parameters:
    -----------
    mat_file : str
        Path to .mat file containing fMRI data
    data_key : str
        Key name in .mat file (e.g., 'Left_data' or 'Right_data')
    output_gifti : str
        Path for output .func.gii file
    vertices_per_hemisphere : int
        Number of vertices (32k surface = 32492)
    
    Returns:
    --------
    output_gifti : str
        Path to created GIFTI file
    """
    print(f"Loading .mat file: {mat_file}")
    
    # Load mat file
    mat_data = sio.loadmat(mat_file)
    
    if data_key not in [k for k in mat_data.keys() if not k.startswith('__')]:
        available_keys = [k for k in mat_data.keys() if not k.startswith('__')]
        raise ValueError(f"Key '{data_key}' not found. Available keys: {available_keys}")
    
    fmri_data = mat_data[data_key]
    print(f"Loaded data with key '{data_key}': shape {fmri_data.shape}")
    
    # Ensure correct orientation: should be (vertices, timepoints)
    if fmri_data.shape[0] != vertices_per_hemisphere:
        if fmri_data.shape[1] == vertices_per_hemisphere:
            print("Transposing data to (vertices, timepoints)")
            fmri_data = fmri_data.T
        else:
            print(f"Warning: Expected {vertices_per_hemisphere} vertices, got shape {fmri_data.shape}")
    
    n_vertices, n_timepoints = fmri_data.shape
    print(f"Data shape: {n_vertices} vertices × {n_timepoints} timepoints")
    
    # Create GIFTI file with one darray per timepoint
    darrays = []
    for t in range(n_timepoints):
        if t % 500 == 0:
            print(f"Creating darray {t}/{n_timepoints}")
        
        darray = nib.gifti.GiftiDataArray(
            data=fmri_data[:, t].astype(np.float32),
            intent='NIFTI_INTENT_NONE',
            datatype='NIFTI_TYPE_FLOAT32'
        )
        darrays.append(darray)
    
    # Create and save GIFTI image
    gii = nib.GiftiImage(darrays=darrays)
    nib.save(gii, output_gifti)
    
    print(f"Saved GIFTI file: {output_gifti}")
    print(f"File size: {os.path.getsize(output_gifti) / (1024**3):.2f} GB")
    
    return output_gifti


def prepare_hcp_movie_data(mat_file_left, mat_file_right, output_dir):
    """
    Prepare both hemispheres from .mat files.
    
    Parameters:
    -----------
    mat_file_left : str
        Path to left hemisphere .mat file
    mat_file_right : str
        Path to right hemisphere .mat file
    output_dir : str
        Directory to save converted files
    
    Returns:
    --------
    gifti_left, gifti_right : tuple
        Paths to converted GIFTI files
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    gifti_left = output_dir / "movie_watching_left.func.gii"
    gifti_right = output_dir / "movie_watching_right.func.gii"
    
    print("Converting LEFT hemisphere...")
    if not gifti_left.exists():
        convert_mat_to_gifti(mat_file_left, 'Left_data', str(gifti_left))
    else:
        print(f"Using existing file: {gifti_left}")
    
    print("\nConverting RIGHT hemisphere...")
    if not gifti_right.exists():
        convert_mat_to_gifti(mat_file_right, 'Right_data', str(gifti_right))
    else:
        print(f"Using existing file: {gifti_right}")
    
    return str(gifti_left), str(gifti_right)

In [ ]:
def compute_geodesic_distance_matrix(surface_file, output_dconn, 
                                     limit_mm=None, roi_metric=None,
                                     corrected_areas=None, naive=False):
    """
    Compute full geodesic distance matrix using wb_command's all-to-all function.
    
    Parameters:
    -----------
    surface_file : str
        Path to surface .surf.gii file
    output_dconn : str
        Output path for dconn file (cifti format)
    limit_mm : float, optional
        Stop at specified distance (vertices beyond get -1)
    roi_metric : str, optional
        ROI metric file - only output distances for ROI vertices
    corrected_areas : str, optional
        Vertex areas for group average surface correction
    naive : bool
        Use only neighbors (not recommended, less accurate)
    
    Returns:
    --------
    output_dconn : str
        Path to output dconn file
    """
    print(f"Computing all-to-all geodesic distances...")
    print(f"Surface: {surface_file}")
    print(f"Output: {output_dconn}")
    
    # Build command
    cmd = [
        'wb_command',
        '-surface-geodesic-distance-all-to-all',
        surface_file,
        output_dconn
    ]
    
    # Add optional parameters
    if roi_metric is not None:
        cmd.extend(['-roi', roi_metric])
        print(f"Using ROI: {roi_metric}")
    
    if limit_mm is not None:
        cmd.extend(['-limit', str(limit_mm)])
        print(f"Distance limit: {limit_mm} mm")
    
    if corrected_areas is not None:
        cmd.extend(['-corrected-areas', corrected_areas])
        print(f"Using corrected areas: {corrected_areas}")
    
    if naive:
        cmd.append('-naive')
        print("Using naive mode (neighbors only)")
    
    # Run command
    print("Running wb_command...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"ERROR: {result.stderr}")
        raise RuntimeError(f"wb_command failed: {result.stderr}")
    
    print(f"Success! Distance matrix saved to {output_dconn}")
    return output_dconn


def load_dconn_as_matrix(dconn_file):
    """
    Load dconn file and convert to numpy matrix.
    
    Parameters:
    -----------
    dconn_file : str
        Path to dconn file from geodesic distance computation
    
    Returns:
    --------
    distance_matrix : np.ndarray
        NxN distance matrix
    """
    print(f"Loading distance matrix from {dconn_file}...")
    
    # Load cifti file
    cifti = nib.load(dconn_file)
    
    # Extract data matrix
    distance_matrix = cifti.get_fdata()
    
    print(f"Distance matrix shape: {distance_matrix.shape}")
    print(f"Distance matrix size in memory: {distance_matrix.nbytes / (1024**3):.2f} GB")
    
    return distance_matrix


def get_k_nearest_neighbors(distance_matrix, k=100):
    """
    Get K nearest neighbors for each vertex from distance matrix.
    
    Parameters:
    -----------
    distance_matrix : np.ndarray or str
        Either NxN distance matrix or path to .npy/.dconn file
    k : int
        Number of nearest neighbors (including center vertex)
    
    Returns:
    --------
    neighbor_indices : np.ndarray
        Shape (N, k) - indices of k nearest neighbors for each vertex
    """
    # Load if path provided
    if isinstance(distance_matrix, (str, Path)):
        if str(distance_matrix).endswith('.npy'):
            distance_matrix = np.load(distance_matrix)
        else:  # assume dconn
            distance_matrix = load_dconn_as_matrix(distance_matrix)
    
    n_vertices = distance_matrix.shape[0]
    print(f"Finding {k} nearest neighbors for {n_vertices} vertices...")
    
    # Get k nearest for each vertex
    neighbor_indices = np.zeros((n_vertices, k), dtype=np.int32)
    
    for i in range(n_vertices):
        if i % 5000 == 0:
            print(f"Progress: {i}/{n_vertices}")
        
        # Get sorted indices by distance
        sorted_idx = np.argsort(distance_matrix[i, :])
        neighbor_indices[i, :] = sorted_idx[:k]
    
    print("Done!")
    return neighbor_indices

In [ ]:
def load_fmri_data_efficient(fmri_file):
    """
    Load fMRI data into memory-efficient format.
    
    Returns:
    --------
    fmri_data : np.ndarray
        Shape (n_vertices, n_timepoints)
    """
    print(f"Loading fMRI data from {fmri_file}...")
    
    fmri = nib.load(fmri_file)
    
    if str(fmri_file).endswith('.func.gii'):
        n_timepoints = len(fmri.darrays)
        n_vertices = fmri.darrays[0].data.shape[0]
        
        fmri_data = np.zeros((n_vertices, n_timepoints), dtype=np.float32)
        for t in range(n_timepoints):
            if t % 500 == 0:
                print(f"  Loading timepoint {t}/{n_timepoints}")
            fmri_data[:, t] = fmri.darrays[t].data
    else:
        fmri_data = fmri.get_fdata().T  # (vertices, timepoints)
    
    print(f"Loaded fMRI data: {fmri_data.shape[0]} vertices × {fmri_data.shape[1]} timepoints")
    return fmri_data


def load_model_rdms(model_file):
    """
    Load pre-computed model RDMs or embeddings. if embeddings, compute RDM. 
    
    Parameters:
    -----------
    model_file : str
        Path to model data (.npy/.npz). 
        Can be either:
        - RDMs: shape (n_timepoints, n_timepoints) - pre-computed dissimilarity matrix
        - Embeddings: shape (n_timepoints, n_features) - will compute RDM from this
    
    Returns:
    --------
    model_rdm : np.ndarray
        RDM for the model (n_timepoints, n_timepoints)
    """
    
    print(f"Loading model data from {model_file}...")
    
    model_data = np.load(model_file)
    
    if model_data.ndim == 2:
        # Check if it's already an RDM (square matrix)
        if model_data.shape[0] == model_data.shape[1]:
            print(f"  Loaded pre-computed RDM: {model_data.shape}")
            return model_data
        else:
            # It's embeddings, compute RDM using COSINE DISSIMILARITY
            print(f"  Computing RDM from embeddings: {model_data.shape}")
            # CHANGED: Use cosine dissimilarity (1 - cosine_similarity)
            cos_sim = cosine_similarity(model_data)
            rdm = 1 - cos_sim
            print(f"  RDM computed with shape: {rdm.shape}")
            return rdm
    else:
        raise ValueError(f"Unexpected model data shape: {model_data.shape}")


def save_as_gifti_metric(data, output_file, metric_name="metric"):
    """
    Save numpy array as GIFTI metric file for Workbench visualization.
    
    Parameters:
    -----------
    data : np.ndarray
        1D array (n_vertices,)
    output_file : str
        Output .func.gii path
    metric_name : str
        Name for the metric
    """
    # Create single darray
    darray = nib.gifti.GiftiDataArray(
        data=data.astype(np.float32),
        intent='NIFTI_INTENT_NONE',
        datatype='NIFTI_TYPE_FLOAT32'
    )
    
    # Add metadata
    meta = nib.gifti.GiftiMetaData()
    meta['Name'] = metric_name
    darray.meta = meta
    
    # Create and save
    gii = nib.GiftiImage(darrays=[darray])
    nib.save(gii, output_file)
    print(f"Saved GIFTI metric: {output_file}")


def compute_searchlight_rsa_map(fmri_file, neighbor_indices, model_rdm,
                                output_file=None, method='spearman',
                                save_pvalues=True, brain_metric='cosine'):  
    """
    Compute RSA correlation map for all searchlights.
    
    Parameters:
    -----------
    fmri_file : str
        Path to fMRI data
    neighbor_indices : np.ndarray
        Shape (n_vertices, k) - neighbor indices for each searchlight
    model_rdm : np.ndarray or str
        Either pre-computed RDM (n_timepoints, n_timepoints) or path to model file
    output_file : str, optional
        Path to save correlation map
    method : str
        'spearman' or 'pearson' or 'kendall' for RSA
    save_pvalues : bool
        Whether to also save p-values
    brain_metric : str
        'correlation' (default) or 'cosine' for brain RDM computation
    """
    # Load fMRI data once
    fmri_data = load_fmri_data_efficient(fmri_file)
    n_vertices, n_timepoints = fmri_data.shape
    
    # Load model RDM if path provided
    if isinstance(model_rdm, (str, Path)):
        model_rdm = load_model_rdms(model_rdm)
    
    print(f"Model RDM shape: {model_rdm.shape}")
    
    # Verify model RDM matches timepoints
    if model_rdm.shape[0] != n_timepoints:
        raise ValueError(
            f"Model RDM timepoints ({model_rdm.shape[0]}) doesn't match "
            f"fMRI timepoints ({n_timepoints})"
        )
    
    # Vectorize model RDM once (upper triangle)
    triu_indices = np.triu_indices(n_timepoints, k=1)
    model_rdm_vec = model_rdm[triu_indices]
    
    # Initialize output arrays
    n_searchlights = neighbor_indices.shape[0]
    rsa_correlations = np.zeros(n_searchlights, dtype=np.float32)
    
    if save_pvalues:
        rsa_pvalues = np.zeros(n_searchlights, dtype=np.float32)
    
    print(f"\nComputing RSA for {n_searchlights} searchlights...")
    print(f"Method: {method}")
    
    # Main loop - compute RSA for each searchlight
    for vertex_idx in range(n_searchlights):
        if vertex_idx % 1000 == 0:
            print(f"Progress: {vertex_idx}/{n_searchlights} ({100*vertex_idx/n_searchlights:.1f}%)")
    
    neighbors = neighbor_indices[vertex_idx, :]
    searchlight_activations = fmri_data[neighbors, :].T  # (n_timepoints, k_neighbors)
    
    # CHANGED: Use specified metric for brain RDM
    if brain_metric == 'cosine': #default
        cos_sim = cosine_similarity(searchlight_activations)
        searchlight_rdm = 1 - cos_sim
    else:  # correlation 
        searchlight_rdm = squareform(pdist(searchlight_activations, metric='correlation'))
        
        # Vectorize searchlight RDM
        searchlight_rdm_vec = searchlight_rdm[triu_indices]
        
        # Compute correlation with model RDM
        if method == 'spearman':
            corr, pval = spearmanr(searchlight_rdm_vec, model_rdm_vec)
        if method == 'pearson':
            corr, pval = pearsonr(searchlight_rdm_vec, model_rdm_vec)
        if method == 'kendall':
            corr, pval = kendalltau(searchlight_rdm_vec, model_rdm_vec)
        
        # Store results
        rsa_correlations[vertex_idx] = corr
        if save_pvalues:
            rsa_pvalues[vertex_idx] = pval
    
    print(f"\nRSA computation complete!")
    print(f"Correlation range: [{np.nanmin(rsa_correlations):.3f}, {np.nanmax(rsa_correlations):.3f}]")
    print(f"Mean correlation: {np.nanmean(rsa_correlations):.3f}")
    
    # Save results
    if output_file:
        output_path = Path(output_file)
        
        # Save correlations
        np.save(output_path.with_suffix('.npy'), rsa_correlations)
        print(f"Saved correlations to {output_path.with_suffix('.npy')}")
        
        if save_pvalues:
            pval_file = output_path.with_name(output_path.stem + '_pvalues.npy')
            np.save(pval_file, rsa_pvalues)
            print(f"Saved p-values to {pval_file}")
        
        # Also save as GIFTI metric for visualization in workbench
        save_as_gifti_metric(rsa_correlations, 
                            str(output_path.with_suffix('.func.gii')),
                            metric_name=f"RSA_{method}")
    
    if save_pvalues:
        return rsa_correlations, rsa_pvalues
    else:
        return rsa_correlations

In [ ]:
def complete_rsa_pipeline(surface_file, fmri_file, model_file, output_dir,
                          n_neighbors=100, limit_mm=None, method='spearman'):
    """
    Complete memory-efficient RSA pipeline.
    
    Parameters:
    -----------
    surface_file : str
        Surface geometry
    fmri_file : str
        fMRI data
    model_file : str
        Model RDM or embeddings (.npy file)
    output_dir : str
        Output directory
    n_neighbors : int
        Searchlight size
    limit_mm : float, optional
        Distance limit in mm
    method : str
        'spearman' or 'pearson'
    
    Returns:
    --------
    rsa_correlations : np.ndarray
        RSA correlation map
    rsa_pvalues : np.ndarray
        P-value map
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print("=" * 70)
    print("MEMORY-EFFICIENT SEARCHLIGHT RSA PIPELINE")
    print("=" * 70)
    
    # Step 1: Compute geodesic distances
    print("\nSTEP 1: Computing geodesic distances...")
    print("-" * 70)
    
    dconn_file = output_dir / "geodesic_distances.dconn.nii"
    
    if not dconn_file.exists():
        compute_geodesic_distance_matrix(
            surface_file=surface_file,
            output_dconn=str(dconn_file),
            limit_mm=limit_mm
        )
    else:
        print(f"Using existing distance file: {dconn_file}")
    
    # Step 2: Get neighbors
    print(f"\nSTEP 2: Finding {n_neighbors} nearest neighbors...")
    print("-" * 70)
    
    neighbors_file = output_dir / f"neighbor_indices_k{n_neighbors}.npy"
    
    if not neighbors_file.exists():
        distance_matrix = load_dconn_as_matrix(str(dconn_file))
        neighbor_indices = get_k_nearest_neighbors(distance_matrix, k=n_neighbors)
        np.save(neighbors_file, neighbor_indices)
        print(f"Saved neighbor indices to {neighbors_file}")
    else:
        print(f"Loading existing neighbor indices: {neighbors_file}")
        neighbor_indices = np.load(neighbors_file)
    
    # Step 3: Compute RSA map (memory efficient!)
    print("\nSTEP 3: Computing RSA correlation map...")
    print("-" * 70)
    
    rsa_output = output_dir / f"rsa_map_{method}_k{n_neighbors}"
    
    rsa_correlations, rsa_pvalues = compute_searchlight_rsa_map(
        fmri_file=fmri_file,
        neighbor_indices=neighbor_indices,
        model_rdm=model_file,
        output_file=str(rsa_output),
        method=method,
        save_pvalues=True
    )
    
    print("\n" + "=" * 70)
    print("PIPELINE COMPLETE!")
    print("=" * 70)
    print(f"Results saved in: {output_dir}")
    print(f"  RSA correlation map: {rsa_output}.func.gii")
    print(f"  RSA p-values: {rsa_output}_pvalues.npy")
    print(f"\nVisualize in Workbench:")
    print(f"  wb_view {surface_file} {rsa_output}.func.gii")
    
    return rsa_correlations, rsa_pvalues

In [ ]:
# Configuration
DATA_DIR = Path("/home/amin/Research/Representation/Movie/data/Setareh")
HCP_DATA_DIR = DATA_DIR / "HCP Data"
SURFACE_DIR = DATA_DIR / "HCP_S1200_GroupAvg_v1"
OUTPUT_DIR = Path("/home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output")

# Input files
mat_file_left = HCP_DATA_DIR / "notmean_left_Meanfile.mat"
mat_file_right = HCP_DATA_DIR / "notmean_right_Meanfile.mat"

surface_left = SURFACE_DIR / "S1200.L.midthickness_MSMAll.32k_fs_LR.surf.gii"
surface_right = SURFACE_DIR / "S1200.R.midthickness_MSMAll.32k_fs_LR.surf.gii"

# ADD TIMING FILE
timing_csv = DATA_DIR / "movie_timing.csv"  # NEED TO CREATE THIS!

# Model files
model_file = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings/pe-av-small/peav_small_16frame_av.npy")

# Parameters
N_NEIGHBORS = 100
LIMIT_MM = None
RSA_METHOD = 'spearman'
TR = 1.0  
BIN_SEC = 2.0 
DELAY_SEC = 4.0 

# ============================================================================
# STEP 0: Process timing and segment fMRI data
# ============================================================================
print("=" * 70)
print("STEP 0: Processing fMRI with timing/segmentation")
print("=" * 70)

# Load timing information
timing_df = pd.read_csv(timing_csv)
print(f"\nTiming information loaded:")
print(timing_df)

converted_dir = OUTPUT_DIR / "converted_data"
converted_dir.mkdir(parents=True, exist_ok=True)

# Process LEFT hemisphere
print("\n--- Processing LEFT hemisphere ---")
mat_data_left = sio.loadmat(mat_file_left)
fmri_left_raw = mat_data_left['Left_data']
if fmri_left_raw.shape[0] != 32492:
    fmri_left_raw = fmri_left_raw.T

print(f"Raw fMRI shape: {fmri_left_raw.shape}")
fmri_left_processed = process_all_video_segments(
    fmri_left_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC
)
print(f"Processed fMRI shape: {fmri_left_processed.shape}")

# Save processed data as GIFTI
gifti_left = converted_dir / "movie_watching_left_processed.func.gii"
print(f"\nSaving processed LEFT hemisphere to GIFTI...")
darrays_left = []
for t in range(fmri_left_processed.shape[1]):
    if t % 500 == 0:
        print(f"  Creating darray {t}/{fmri_left_processed.shape[1]}")
    darray = nib.gifti.GiftiDataArray(
        data=fmri_left_processed[:, t].astype(np.float32),
        intent='NIFTI_INTENT_NONE',
        datatype='NIFTI_TYPE_FLOAT32'
    )
    darrays_left.append(darray)
gii_left = nib.GiftiImage(darrays=darrays_left)
nib.save(gii_left, gifti_left)
print(f"Saved: {gifti_left}")

# Process RIGHT hemisphere (same process)
print("\n--- Processing RIGHT hemisphere ---")
mat_data_right = sio.loadmat(mat_file_right)
fmri_right_raw = mat_data_right['Right_data']
if fmri_right_raw.shape[0] != 32492:
    fmri_right_raw = fmri_right_raw.T

print(f"Raw fMRI shape: {fmri_right_raw.shape}")
fmri_right_processed = process_all_video_segments(
    fmri_right_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC
)
print(f"Processed fMRI shape: {fmri_right_processed.shape}")

gifti_right = converted_dir / "movie_watching_right_processed.func.gii"
print(f"\nSaving processed RIGHT hemisphere to GIFTI...")
darrays_right = []
for t in range(fmri_right_processed.shape[1]):
    if t % 500 == 0:
        print(f"  Creating darray {t}/{fmri_right_processed.shape[1]}")
    darray = nib.gifti.GiftiDataArray(
        data=fmri_right_processed[:, t].astype(np.float32),
        intent='NIFTI_INTENT_NONE',
        datatype='NIFTI_TYPE_FLOAT32'
    )
    darrays_right.append(darray)
gii_right = nib.GiftiImage(darrays=darrays_right)
nib.save(gii_right, gifti_right)
print(f"Saved: {gifti_right}")


# ============================================================================
# STEP 1-3: Run RSA pipeline for LEFT hemisphere
# ============================================================================
print("\n" + "=" * 70)
print("PROCESSING LEFT HEMISPHERE")
print("=" * 70)

rsa_corr_left, rsa_pval_left = complete_rsa_pipeline(
    surface_file=str(surface_left),
    fmri_file=str(gifti_left),
    model_file=str(model_file),
    output_dir=str(OUTPUT_DIR / "left_hemisphere"),
    n_neighbors=N_NEIGHBORS,
    limit_mm=LIMIT_MM,
    method=RSA_METHOD
)

# ============================================================================
# STEP 1-3: Run RSA pipeline for RIGHT hemisphere
# ============================================================================
print("\n" + "=" * 70)
print("PROCESSING RIGHT HEMISPHERE")
print("=" * 70)

rsa_corr_right, rsa_pval_right = complete_rsa_pipeline(
    surface_file=str(surface_right),
    fmri_file=str(gifti_right),
    model_file=str(model_file),
    output_dir=str(OUTPUT_DIR / "right_hemisphere"),
    n_neighbors=N_NEIGHBORS,
    limit_mm=LIMIT_MM,
    method=RSA_METHOD
)

# ============================================================================
# SUMMARY & STATISTICS
# ============================================================================
print("\n" + "=" * 70)
print("ALL PROCESSING COMPLETE!")
print("=" * 70)

print(f"\nOutput directory: {OUTPUT_DIR}")

print(f"\n{'='*70}")
print("LEFT HEMISPHERE RESULTS:")
print(f"{'='*70}")
print(f"  RSA correlation range: [{rsa_corr_left.min():.4f}, {rsa_corr_left.max():.4f}]")
print(f"  Mean correlation: {rsa_corr_left.mean():.4f} ± {rsa_corr_left.std():.4f}")
print(f"  Median correlation: {np.median(rsa_corr_left):.4f}")
print(f"  Significant vertices (p < 0.05): {(rsa_pval_left < 0.05).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.05).sum()/len(rsa_pval_left):.1f}%)")
print(f"  Significant vertices (p < 0.01): {(rsa_pval_left < 0.01).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.01).sum()/len(rsa_pval_left):.1f}%)")
print(f"  Significant vertices (p < 0.001): {(rsa_pval_left < 0.001).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.001).sum()/len(rsa_pval_left):.1f}%)")

print(f"\n{'='*70}")
print("RIGHT HEMISPHERE RESULTS:")
print(f"{'='*70}")
print(f"  RSA correlation range: [{rsa_corr_right.min():.4f}, {rsa_corr_right.max():.4f}]")
print(f"  Mean correlation: {rsa_corr_right.mean():.4f} ± {rsa_corr_right.std():.4f}")
print(f"  Median correlation: {np.median(rsa_corr_right):.4f}")
print(f"  Significant vertices (p < 0.05): {(rsa_pval_right < 0.05).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.05).sum()/len(rsa_pval_right):.1f}%)")
print(f"  Significant vertices (p < 0.01): {(rsa_pval_right < 0.01).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.01).sum()/len(rsa_pval_right):.1f}%)")
print(f"  Significant vertices (p < 0.001): {(rsa_pval_right < 0.001).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.001).sum()/len(rsa_pval_right):.1f}%)")

# ============================================================================
# VISUALIZATION COMMANDS
# ============================================================================
print(f"\n{'='*70}")
print("VISUALIZATION IN CONNECTOME WORKBENCH:")
print(f"{'='*70}")
print("\nLeft hemisphere:")
print(f"  wb_view {surface_left} \\")
print(f"    {OUTPUT_DIR}/left_hemisphere/rsa_map_{RSA_METHOD}_k{N_NEIGHBORS}.func.gii")

print("\nRight hemisphere:")
print(f"  wb_view {surface_right} \\")
print(f"    {OUTPUT_DIR}/right_hemisphere/rsa_map_{RSA_METHOD}_k{N_NEIGHBORS}.func.gii")

print("\nBoth hemispheres together:")
print(f"  wb_view {surface_left} \\")
print(f"    {OUTPUT_DIR}/left_hemisphere/rsa_map_{RSA_METHOD}_k{N_NEIGHBORS}.func.gii \\")
print(f"    {surface_right} \\")
print(f"    {OUTPUT_DIR}/right_hemisphere/rsa_map_{RSA_METHOD}_k{N_NEIGHBORS}.func.gii")

# ============================================================================
# SAVE COMBINED SUMMARY
# ============================================================================
summary_file = OUTPUT_DIR / "rsa_analysis_summary.txt"
with open(summary_file, 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("SEARCHLIGHT RSA ANALYSIS SUMMARY\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Searchlight size: {N_NEIGHBORS} neighbors\n")
    f.write(f"RSA method: {RSA_METHOD}\n")
    f.write(f"Distance limit: {LIMIT_MM if LIMIT_MM else 'None (unlimited)'}\n\n")
    
    f.write("LEFT HEMISPHERE:\n")
    f.write(f"  Mean correlation: {rsa_corr_left.mean():.4f}\n")
    f.write(f"  Significant (p<0.05): {(rsa_pval_left < 0.05).sum()} vertices\n\n")
    
    f.write("RIGHT HEMISPHERE:\n")
    f.write(f"  Mean correlation: {rsa_corr_right.mean():.4f}\n")
    f.write(f"  Significant (p<0.05): {(rsa_pval_right < 0.05).sum()} vertices\n")

print(f"\nSummary saved to: {summary_file}")

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE! 🎉")
print("=" * 70)